In [9]:
from pathlib import Path
import cv2

import numpy as np

In [11]:
baseline = 0.72

#Camera Matrix

camera_matrix = np.array([[7.188560e+02, 0.000000e+00, 6.071928e+02], [0, 7.188560e+02, 1.852157e+02], [0, 0, 1]])

projection_left = camera_matrix.dot(np.hstack((np.eye(3), np.zeros((3, 1)))))

projection_right = camera_matrix.dot(np.hstack((np.eye(3), np.array([[-baseline, 0, 0]]).T)))

In [8]:
left_images = Path('left').glob('*.png')
right_images = Path('right').glob('*.png')

image_1_left = cv2.imread(next(left_images))
image_1_right = cv2.imread(next(right_images))

image_2_left = cv2.imread(next(left_images))
image_2_right = cv2.imread(next(right_images))

In [ ]:
sift = cv2.SIFT.create()

gray_1_left = cv2.cvtColor(image_1_left, cv2.COLOR_BGR2GRAY)
gray_1_right = cv2.cvtColor(image_1_right, cv2.COLOR_BGR2GRAY)

keypoints_left, descriptors_left = sift.detectAndCompute(gray_1_left, None)
keypoints_right, descriptors_right = sift.detectAndCompute(gray_1_right, None)

brute_force_matcher = cv2.BFMatcher()
matches = brute_force_matcher.match(descriptors_left, descriptors_right)

matches = sorted(matches, key=lambda x: x.distance)

points1 = []
points2 = []

for m in matches:
    points1.append(keypoints_left[m.queryIdx].pt)
    points2.append(keypoints_right[m.trainIdx].pt)

points1 = np.int32(points1)
points2 = np.int32(points2)

F, mask = cv2.findFundamentalMat(points1, points2, cv2.FM_RANSAC)

# We select only inlier points
points1 = points1[mask.ravel() == 1]
points2 = points2[mask.ravel() == 1]

WindowsPath('left/000300.png')